In [1]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections
import matplotlib.pyplot as plt
# %matplotlib inline

import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True

In [2]:
df_sentiment = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTime.pkl")

In [3]:
df_sentiment

,Time,Date,Hour,sentiment,compound,neg,neu,pos,polarity,subjectivity,polarityB,Articles,Title,Text,Company,MarketCap
0,2020-06-01 04:26:00-05:00,2020-06-01,4,3.624046e+10,-0.9998,0.911,0.089,0.000,-0.155556,0.288889,Negative,GM'S SOUTH KOREA UNIT SAYS MAY SALES DOWN 40% ...,GM'S SOUTH KOREA UNIT SAYS MAY SALES DOWN 40% Y/Y,GM'S SOUTH KOREA UNIT: MAY SALES DOWN 40% Y/Y ...,GM,3.978096e+10
1,2020-06-01 04:26:00-05:00,2020-06-01,4,5.366679e+10,0.8074,0.059,0.742,0.198,0.000000,0.000000,Neutral,ASTRAZENECA-MERCK LYNPARZA GETS RECOMMENDATION...,ASTRAZENECA-MERCK LYNPARZA GETS RECOMMENDATION...,ASTRAZENECA PLC - LYNPARZA EU CHMP OPINION IN...,MRK,2.086114e+11
2,2020-06-01 08:26:00-05:00,2020-06-01,8,7.948280e+10,0.9993,0.009,0.467,0.524,0.107143,0.571429,Positive,LILLY BEGINS WORLD'S FIRST STUDY OF POTENTIAL ...,LILLY BEGINS WORLD'S FIRST STUDY OF POTENTIAL ...,LILLY BEGINS WORLD'S FIRST STUDY OF A POTENTIA...,LLY,1.491234e+11
3,2020-06-01 09:26:00-05:00,2020-06-01,9,5.787987e+10,0.9999,0.000,0.186,0.814,0.068182,0.727273,Positive,"REGENERON, INTELLIA EXPAND COLLABORATION TO DE...","REGENERON, INTELLIA EXPAND COLLABORATION TO DE...",REGENERON AND INTELLIA THERAPEUTICS EXPAND COL...,REGN,7.110549e+10
4,2020-06-01 09:26:00-05:00,2020-06-01,9,5.681578e+09,0.4466,0.000,0.956,0.044,-0.083333,0.083333,Negative,COMPUGEN REPORTS FDA CLEARANCE OF IND APPLICAT...,COMPUGEN REPORTS FDA CLEARANCE OF IND APPLICAT...,COMPUGEN ANNOUNCES FDA CLEARANCE OF IND APPLIC...,BMY,1.291268e+11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30891,2022-06-01 19:13:00-05:00,2022-06-01,19,1.268432e+10,-0.9255,0.148,0.852,0.000,0.034545,0.321818,Positive,NFLX LAWSUIT: THE LAW OFFICES OF VINCENT WONG ...,NFLX LAWSUIT: THE LAW OFFICES OF VINCENT WONG ...,"NEW YORK, NEW YORK--(NEWSFILE CORP. - JUNE 1, ...",NFLX,8.570489e+10
30892,2022-06-01 19:17:00-05:00,2022-06-01,19,1.854406e+10,0.9998,0.000,0.265,0.735,-0.014286,0.293132,Negative,ROCKWELL AUTOMATION DECLARES QUARTERLY DIVIDEN...,ROCKWELL AUTOMATION DECLARES QUARTERLY DIVIDEN...,MILWAUKEE--(BUSINESS WIRE)--THE BOARD OF DIREC...,ROK,2.523002e+10
30893,2022-06-01 19:43:00-05:00,2022-06-01,19,3.289954e+11,0.8109,0.098,0.735,0.168,0.114502,0.280087,Positive,AMZN LAWSUIT ALERT: LEVI & KORSINSKY NOTIFIES ...,AMZN LAWSUIT ALERT: LEVI & KORSINSKY NOTIFIES ...,"NEW YORK, NEW YORK--(NEWSFILE CORP. - JUNE 1, ...",AMZN,1.238062e+12
30894,2022-06-01 22:56:00-05:00,2022-06-01,22,7.572691e+09,0.9055,0.035,0.792,0.173,0.178788,0.601515,Positive,TAIWAN NATIONAL POLICE TO BOLSTER EMERGENCY RE...,TAIWAN NATIONAL POLICE TO BOLSTER EMERGENCY RE...,TAIPEI--(BUSINESS WIRE)--MOTOROLA SOLUTIONS (N...,MSI,3.640717e+10


In [4]:
df_sentiment.index = pd.to_datetime(df_sentiment.index)
df_sentiment.Date = pd.to_datetime(df_sentiment.Date)
df_sentiment.sort_values(by="Time",inplace=True)
df_sentiment.index = df_sentiment.Time.to_period(freq='T').index

In [5]:
# df_sentiment = df_sentiment.groupby("Date").sum()
df_sentiment["sentiment"] = df_sentiment["sentiment"]/df_sentiment["MarketCap"]

In [6]:
df_sentiment.index = df_sentiment.Time

In [7]:
df_sentiment.Date = df_sentiment.index

In [8]:
df_sentiment.rename(columns={"vix":"VIX",
                "vix(t-1)":"VIX(t-1)",
                "vix_ln_return":"VIX return",
                "sentiment":"Sentiment",
                "Sentiment_ln_return":"Sentiment return",
                "Sentiment_ln_return(t-1)":"Sentiment return(t-1)",
                "pos":"Positive",
                "pos(t-1)":"Positive(t-1)",
                "pos(t-2)":"Positive(t-2)",
                "pos_ln_return":"Positive return",
                "pos_ln_return(t-1)":"Positive return(t-1)",
                "neg":"Negative",
                "neg(t-1)":"Negative(t-1)", 
                "neg(t-2)":"Negative(t-2)",
                "neg_ln_return":"Negative return", 
                "neg_ln_return(t-1)":"Negative return(t-1)", 
                "neu":"Neutral", 
                "neu(t-1)":"Neutral(t-1)",
                "neu(t-2)":"Neutral(t-2)",
                "neu_ln_return":"Neutral return",
                "neu_ln_return(t-1)":"Neutral return(t-1)",
                "compound":"Compound",
                "compound(t-1)":"Compound(t-1)",
                "compound(t-2)":"Compound(t-2)",
                "compound_ln_return":"Compound return",
                "compound_ln_return(t-1)":"Compound return(t-1)",
                "polarity":"Polarity",
                "polarity(t-1)":"Polarity(t-1)",
                "polarityB":"PolarityB",
                "subjectivity":"Subjectivity",
                "subjectivity(t-1)":"Subjectivity(t-1)",
                "PressReleaseCount_ln_return":"PressReleaseCount return",
                "PressReleaseCount_ln_return(t-1)":"PressReleaseCount return(t-1)",
                "WordCount_ln_return":"WordCount return",
                "WordCount_ln_return(t-1)":"WordCount return(t-1)",
                "day":"Day of Week",
                "month":"Month",
                "month-year":"Month-Year",
                "quarter":"Quarter",
                "year":"Year",
                "GI_Sentiment_ln_return":"GI_Sentiment return",
                "GI_Negative_ln_return":"GI_Negative return",
                "GI_Positive_ln_return":"GI_Positive return"}
                ,inplace=True)

In [9]:
# df_sentiment.drop(columns=["compound", "Negative", "Positive", "neu", "Articles", "Title", "Text", "Company"], inplace=True)
# df_sentiment.drop(columns=["Company"], inplace=True)


df_sentiment = df_sentiment.ffill()

df_sentiment["Sentiment(t-1)"] = df_sentiment["Sentiment"].shift(periods=1)
df_sentiment["Sentiment(t-2)"] = df_sentiment["Sentiment"].shift(periods=2)
df_sentiment["Sentiment(t-1)"] = df_sentiment["Sentiment(t-1)"].ffill()
df_sentiment["Sentiment(t-2)"] = df_sentiment["Sentiment(t-2)"].ffill()

# SENTIMENT RETURN
# df_sentiment["Sentiment_return"]=(df_sentiment["Sentiment(t-1)"]-df_sentiment["Sentiment(t-2)"])/df_sentiment["Sentiment(t-2)"]
# SENTIMENT LN RETURN
df_sentiment["Sentiment return"]=np.log(df_sentiment["Sentiment"]/df_sentiment["Sentiment(t-1)"])
df_sentiment["Sentiment return(t-1)"]=np.log(df_sentiment["Sentiment(t-1)"]/df_sentiment["Sentiment(t-2)"])

# SENTIMENT EWM
df_sentiment["Sentiment_ewm10"]=df_sentiment["Sentiment"].ewm(10).mean()
df_sentiment["Sentiment_ewm20"]=df_sentiment["Sentiment"].ewm(20).mean()

# POSITIVE LN RETURN
df_sentiment["Positive(t-1)"] = df_sentiment["Positive"].shift(periods=1)
df_sentiment["Positive(t-2)"] = df_sentiment["Positive"].shift(periods=2)
df_sentiment["Positive(t-1)"] = df_sentiment["Positive(t-1)"].ffill()
df_sentiment["Positive(t-2)"] = df_sentiment["Positive(t-2)"].ffill()
df_sentiment["Positive return"]=np.log(df_sentiment["Positive"]/df_sentiment["Positive(t-1)"])
df_sentiment["Positive return(t-1)"]=np.log(df_sentiment["Positive(t-1)"]/df_sentiment["Positive(t-2)"])

# NEGATIVE LN RETURN
df_sentiment["Negative(t-1)"] = df_sentiment["Negative"].shift(periods=1)
df_sentiment["Negative(t-2)"] = df_sentiment["Negative"].shift(periods=2)
df_sentiment["Negative(t-1)"] = df_sentiment["Negative(t-1)"].ffill()
df_sentiment["Negative(t-2)"] = df_sentiment["Negative(t-2)"].ffill()
df_sentiment["Negative return"]=np.log(df_sentiment["Negative"]/df_sentiment["Negative(t-1)"])
df_sentiment["Negative return(t-1)"]=np.log(df_sentiment["Negative(t-1)"]/df_sentiment["Negative(t-2)"])

# # NEUTRAL LN RETURN
# df_sentiment["Neutral(t-1)"] = df_sentiment["Neutral"].shift(periods=1)
# df_sentiment["Neutral(t-2)"] = df_sentiment["Neutral"].shift(periods=2)
# df_sentiment["Neutral(t-1)"] = df_sentiment["Neutral(t-1)"].ffill()
# df_sentiment["Neutral(t-2)"] = df_sentiment["Neutral(t-2)"].ffill()
# df_sentiment["Neutral return"]=np.log(df_sentiment["Neutral"]/df_sentiment["Neutral(t-1)"])
# df_sentiment["Neutral return(t-1)"]=np.log(df_sentiment["Neutral(t-1)"]/df_sentiment["Neutral(t-2)"])

# # COMPOUND LN RETURN
# df_sentiment["Compound(t-1)"] = df_sentiment["Compound"].shift(periods=1)
# df_sentiment["Compound(t-2)"] = df_sentiment["Compound"].shift(periods=2)
# df_sentiment["Compound(t-1)"] = df_sentiment["Compound(t-1)"].ffill()
# df_sentiment["Compound(t-2)"] = df_sentiment["Compound(t-2)"].ffill()
# df_sentiment["Compound return"]=np.log(df_sentiment["Compound"]/df_sentiment["Compound(t-1)"])
# df_sentiment["Compound return(t-1)"]=np.log(df_sentiment["Compound(t-1)"]/df_sentiment["Compound(t-2)"])

df_sentiment["Polarity(t-1)"] = df_sentiment["Polarity"].shift(periods=1)
df_sentiment["Subjectivity(t-1)"] = df_sentiment["Subjectivity"].shift(periods=1)

df_sentiment['WordCount'] = df_sentiment['Articles'].apply(lambda x:len(str(x).split())) #Number Of words in main text
df_sentiment["WordCount(t-1)"] = df_sentiment["WordCount"].shift(periods=1)
df_sentiment["WordCount(t-2)"] = df_sentiment["WordCount"].shift(periods=2)
df_sentiment["WordCount(t-1)"] = df_sentiment["WordCount(t-1)"].ffill()
df_sentiment["WordCount(t-2)"] = df_sentiment["WordCount(t-2)"].ffill()
df_sentiment["WordCount return"]=np.log(df_sentiment["WordCount"]/df_sentiment["WordCount(t-1)"])

In [10]:
(df_sentiment["Positive return"]!=-np.inf).sum()

27122

In [11]:
(df_sentiment["Positive return"]>0 ).sum()

14905

In [12]:
df_sentiment

,Time,Date,Hour,Sentiment,Compound,Negative,Neutral,Positive,Polarity,Subjectivity,...,Negative(t-1),Negative(t-2),Negative return,Negative return(t-1),Polarity(t-1),Subjectivity(t-1),WordCount,WordCount(t-1),WordCount(t-2),WordCount return
Time,,,,,,,,,,,,,,,,,,,,,
2020-06-01 04:26:00-05:00,2020-06-01 04:26:00-05:00,2020-06-01 04:26:00-05:00,4,0.911000,-0.9998,0.911,0.089,0.000,-0.155556,0.288889,...,NaN,NaN,NaN,NaN,NaN,NaN,22,NaN,NaN,NaN
2020-06-01 04:26:00-05:00,2020-06-01 04:26:00-05:00,2020-06-01 04:26:00-05:00,4,0.257257,0.8074,0.059,0.742,0.198,0.000000,0.000000,...,0.911,NaN,-2.737005,NaN,-0.155556,0.288889,65,22.0,NaN,1.083345
2020-06-01 08:26:00-05:00,2020-06-01 08:26:00-05:00,2020-06-01 08:26:00-05:00,8,0.533000,0.9993,0.009,0.467,0.524,0.107143,0.571429,...,0.059,0.911,-1.880313,-2.737005,0.000000,0.000000,110,65.0,22.0,0.526093
2020-06-01 09:26:00-05:00,2020-06-01 09:26:00-05:00,2020-06-01 09:26:00-05:00,9,0.814000,0.9999,0.000,0.186,0.814,0.068182,0.727273,...,0.009,0.059,-inf,-1.880313,0.107143,0.571429,89,110.0,65.0,-0.211844
2020-06-01 09:26:00-05:00,2020-06-01 09:26:00-05:00,2020-06-01 09:26:00-05:00,9,0.044000,0.4466,0.000,0.956,0.044,-0.083333,0.083333,...,0.000,0.009,NaN,-inf,0.068182,0.727273,67,89.0,110.0,-0.283944
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-06-01 19:13:00-05:00,2022-06-01 19:13:00-05:00,2022-06-01 19:13:00-05:00,19,0.148000,-0.9255,0.148,0.852,0.000,0.034545,0.321818,...,0.099,0.000,0.402092,inf,0.137403,0.336104,93,93.0,29.0,0.000000
2022-06-01 19:17:00-05:00,2022-06-01 19:17:00-05:00,2022-06-01 19:17:00-05:00,19,0.735000,0.9998,0.000,0.265,0.735,-0.014286,0.293132,...,0.148,0.099,-inf,0.402092,0.034545,0.321818,85,93.0,93.0,-0.089948
2022-06-01 19:43:00-05:00,2022-06-01 19:43:00-05:00,2022-06-01 19:43:00-05:00,19,0.265734,0.8109,0.098,0.735,0.168,0.114502,0.280087,...,0.000,0.148,inf,-inf,-0.014286,0.293132,93,85.0,93.0,0.089948


# VIX

In [13]:
import datetime, pytz
from time import timezone

df_vix = pd.read_csv("/Users/Farid Srouji/Documents/Research/master/Data/VIX/vix-1m.csv",delimiter=";", decimal=",",names=["Date", "Minute", "Open", "High","Low","Close","Volume"])
# tz_chicago = pytz.timezone('America/Chicago')
df_vix.index = pd.to_datetime(df_vix['Date'] + ' ' + df_vix['Minute'], utc=True)\
                      .map(lambda x: x.tz_convert('America/Chicago'))#,dtype='datetime64[ns, Chi/Kolkata]')#tzinfo=tz_chicago)
df_vix["Open"] = df_vix["Open"].astype(str).astype(float)
df_vix["High"] = df_vix["High"].astype(str).astype(float)
df_vix["Low"] = df_vix["Low"].astype(str).astype(float)
df_vix["Close"] = df_vix["Close"].astype(str).astype(float)
df_vix["Volume"] = df_vix["Volume"].astype(str).astype(float)

df_vix = df_vix["2020-06-01":"2022-06-01"]
df_vix.Date = df_vix.index

In [14]:
df_vix.Date = pd.to_datetime(df_vix.Date)


In [15]:
df_vix

,Date,Minute,Open,High,Low,Close,Volume
2020-06-01 00:01:00-05:00,2020-06-01 00:01:00-05:00,05:01:00,15.81,15.81,15.81,15.81,1.0
2020-06-01 00:02:00-05:00,2020-06-01 00:02:00-05:00,05:02:00,15.81,15.81,15.81,15.81,1.0
2020-06-01 00:03:00-05:00,2020-06-01 00:03:00-05:00,05:03:00,15.80,15.80,15.80,15.80,27.0
2020-06-01 00:04:00-05:00,2020-06-01 00:04:00-05:00,05:04:00,15.85,15.85,15.81,15.85,11.0
2020-06-01 00:05:00-05:00,2020-06-01 00:05:00-05:00,05:05:00,15.88,15.88,15.88,15.88,1.0
...,...,...,...,...,...,...,...
2022-05-09 18:23:00-05:00,2022-05-09 18:23:00-05:00,23:23:00,25.45,25.47,25.45,25.47,16.0
2022-05-09 18:24:00-05:00,2022-05-09 18:24:00-05:00,23:24:00,25.45,25.45,25.45,25.45,4.0
2022-05-09 18:49:00-05:00,2022-05-09 18:49:00-05:00,23:49:00,25.45,25.45,25.45,25.45,4.0
2022-05-09 18:55:00-05:00,2022-05-09 18:55:00-05:00,23:55:00,25.50,25.50,25.50,25.50,1.0


In [16]:
df_vix.index = df_vix.Date.to_period(freq='T').index

In [17]:
df_vix.index=df_vix['Date'].dt.strftime('%Y-%m-%d %H')


In [18]:
df_vix

,Date,Minute,Open,High,Low,Close,Volume
Date,,,,,,,
2020-06-01 00,2020-06-01 00:01:00-05:00,05:01:00,15.81,15.81,15.81,15.81,1.0
2020-06-01 00,2020-06-01 00:02:00-05:00,05:02:00,15.81,15.81,15.81,15.81,1.0
2020-06-01 00,2020-06-01 00:03:00-05:00,05:03:00,15.80,15.80,15.80,15.80,27.0
2020-06-01 00,2020-06-01 00:04:00-05:00,05:04:00,15.85,15.85,15.81,15.85,11.0
2020-06-01 00,2020-06-01 00:05:00-05:00,05:05:00,15.88,15.88,15.88,15.88,1.0
...,...,...,...,...,...,...,...
2022-05-09 18,2022-05-09 18:23:00-05:00,23:23:00,25.45,25.47,25.45,25.47,16.0
2022-05-09 18,2022-05-09 18:24:00-05:00,23:24:00,25.45,25.45,25.45,25.45,4.0
2022-05-09 18,2022-05-09 18:49:00-05:00,23:49:00,25.45,25.45,25.45,25.45,4.0


In [19]:
len(df_vix.index.unique())

11549

In [20]:
df_vix.drop(columns={"Open","High","Low"},inplace=True)
df_vix.rename(columns={"Close":"VIX"},inplace=True)
# df_vix.index = pd.to_datetime(df_vix.index)
# df_vix["Date"]= pd.to_datetime(df_vix.index)
df_vix = df_vix.loc[df_vix["Date"]>"2020-05-31"]

df_vix["VIX(t-1)"] = df_vix["VIX"].shift(periods=1)
df_vix["VIX(t-1)"].ffill(inplace=True)

df_vix["vix_ln_return"]=np.log(df_vix["VIX"]/df_vix["VIX(t-1)"])
df_vix["vix_ln_return"].ffill(inplace=True)

In [21]:
df_vix.drop(columns={"Date"},inplace=True)

In [22]:
df = df_vix.merge(df_sentiment,right_index=True, left_index=True)
df = df.ffill()
df = df.sort_index()

In [35]:
df_vix

,Minute,VIX,Volume,VIX(t-1),vix_ln_return
Date,,,,,
2020-06-01 00,05:01:00,15.81,1.0,NaN,NaN
2020-06-01 00,05:02:00,15.81,1.0,15.81,0.000000
2020-06-01 00,05:03:00,15.80,27.0,15.81,-0.000633
2020-06-01 00,05:04:00,15.85,11.0,15.80,0.003160
2020-06-01 00,05:05:00,15.88,1.0,15.85,0.001891
...,...,...,...,...,...
2022-05-09 18,23:23:00,25.47,16.0,25.50,-0.001177
2022-05-09 18,23:24:00,25.45,4.0,25.47,-0.000786
2022-05-09 18,23:49:00,25.45,4.0,25.45,0.000000


In [23]:
df.dropna(inplace=True)

mx = df.loc[df['Sentiment'] != np.inf, 'Sentiment'].max()
df['Sentiment'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['Sentiment'] != -np.inf, 'Sentiment'].min()
df['Sentiment'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['Sentiment'] != np.inf, 'Sentiment'].max()
df['Sentiment'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['Sentiment'] != -np.inf, 'Sentiment'].min()
df['Sentiment'].replace(-np.inf,mn,inplace=True)

In [24]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df['Year'] = [d.year for d in df.Date]
df['Month'] = [d.strftime('%b') for d in df.Date]
df['Day of Week'] = [d.day_name() for d in df.Date]
years = df['Year'].unique()
df['Month-Year'] = [d.strftime('%b') + str(d.year) for d in df.Date]
df['Quarter'] = pd.PeriodIndex(df.Date, freq='Q')

mx = df.loc[df['Sentiment return'] != np.inf, 'Sentiment return'].max()
df['Sentiment return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['Sentiment return'] != -np.inf, 'Sentiment return'].min()
df['Sentiment return'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['vix_ln_return'] != np.inf, 'vix_ln_return'].max()
df['vix_ln_return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['vix_ln_return'] != -np.inf, 'vix_ln_return'].min()
df['vix_ln_return'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['Positive return'] != np.inf, 'Positive return'].max()
df['Positive return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['Positive return'] != -np.inf, 'Positive return'].min()
df['Positive return'].replace(-np.inf,mn,inplace=True)

mx = df.loc[df['Negative return'] != np.inf, 'Negative return'].max()
df['Negative return'].replace(np.inf,mx,inplace=True)
mn = df.loc[df['Negative return'] != -np.inf, 'Negative return'].min()
df['Negative return'].replace(-np.inf,mn,inplace=True)

In [25]:
df.columns

Index(['Minute', 'VIX', 'Volume', 'VIX(t-1)', 'vix_ln_return', 'Time', 'Date',
       'Hour', 'Sentiment', 'Compound', 'Negative', 'Neutral', 'Positive',
       'Polarity', 'Subjectivity', 'PolarityB', 'Articles', 'Title', 'Text',
       'Company', 'MarketCap', 'Sentiment(t-1)', 'Sentiment(t-2)',
       'Sentiment return', 'Sentiment return(t-1)', 'Sentiment_ewm10',
       'Sentiment_ewm20', 'Positive(t-1)', 'Positive(t-2)', 'Positive return',
       'Positive return(t-1)', 'Negative(t-1)', 'Negative(t-2)',
       'Negative return', 'Negative return(t-1)', 'Polarity(t-1)',
       'Subjectivity(t-1)', 'WordCount', 'WordCount(t-1)', 'WordCount(t-2)',
       'WordCount return', 'Year', 'Month', 'Day of Week', 'Month-Year',
       'Quarter'],
      dtype='object')

In [27]:
df.index = df.Time

In [28]:
df.sort_index(inplace=True)

In [29]:
# df[["VIX","Sentiment"]].plot.scatter(x = 'Sentiment', y = 'VIX', c = 'red');

In [30]:
df.rename(columns={"vix":"VIX",
                "vix(t-1)":"VIX(t-1)",
                "vix_ln_return":"VIX return",
                "PressReleaseCount_ln_return":"PressReleaseCount return",
                "PressReleaseCount_ln_return(t-1)":"PressReleaseCount return(t-1)",
                "WordCount_ln_return":"WordCount return",
                "WordCount_ln_return(t-1)":"WordCount return(t-1)",
                "day":"Day of Week",
                "month":"Month",
                "month-year":"Month-Year",
                "quarter":"Quarter",
                "year":"Year",
                "GI_Sentiment_ln_return":"GI_Sentiment return",
                "GI_Negative_ln_return":"GI_Negative return",
                "GI_Positive_ln_return":"GI_Positive return"}
                ,inplace=True)

In [31]:
dfMinute = df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/dfMinute.pkl")

In [32]:
dfMinute